In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.v_top_tags_by_quarter AS
SELECT *
FROM (
    SELECT
        post_year,
        post_quarter,
        tag,
        tag_count,
        distinct_users,
        RANK() OVER (
            PARTITION BY post_year, post_quarter
            ORDER BY tag_count DESC
        ) AS rnk
    FROM workspace.default.pinterest_tag_counts_gold
)
WHERE tag_count >= 2;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.v_tag_trend AS
SELECT
  post_year,
  post_quarter,
  (post_year * 10 + post_quarter) AS year_q,
  tag,
  tag_count,
  distinct_users
FROM workspace.default.pinterest_tag_counts_gold;

In [0]:
%sql
SELECT
    COUNT(*) AS total_tag_rows,
    COUNT(DISTINCT tag) AS unique_tags
FROM workspace.default.pinterest_tag_counts_gold;

In [0]:
from pyspark.sql import functions as F
GOLD_TABLE   = "workspace.default.pinterest_tag_counts_gold"

audit = spark.createDataFrame(
    [(spark.sql(f"select count(*) from {GOLD_TABLE}").collect()[0][0],)],
    ["gold_row_count"]
).withColumn("run_ts", F.current_timestamp())

audit.write.mode("append").saveAsTable("workspace.default.pipeline_run_audit")